# Car Price Prediction - Machine Learning Pipeline
This notebook demonstrates end-to-end exploratory data analysis (EDA), data preprocessing, feature engineering, and model training comparing:
1. **Linear Regression**
2. **Decision Tree Regressor**
3. **Random Forest Regressor**

The best-performing model is serialized for production serving.

In [1]:
import os
import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

print('Libraries imported successfully!')

## 1. Load and Inspect Dataset

In [2]:
df = pd.read_csv('../data/car_data.csv')
print(f'Dataset Shape: {df.shape}')
df.head()

## 2. Preprocessing & Outlier Handling

In [3]:
# Missing value checks
print('Missing values per column:\n', df.isnull().sum())

# Impute numerical medians
df['Present_Price'] = df['Present_Price'].fillna(df['Present_Price'].median())
df['Kms_Driven'] = df['Kms_Driven'].fillna(df['Kms_Driven'].median())
df['Fuel_Type'] = df['Fuel_Type'].fillna(df['Fuel_Type'].mode()[0])

# Feature engineering
current_year = 2024
df['Car_Age'] = current_year - df['Year']
df['Brand'] = df['Car_Name'].apply(lambda x: x.split()[0] if isinstance(x, str) else 'Other')

# Cap high-mileage outliers at 99th percentile
kms_99 = df['Kms_Driven'].quantile(0.99)
df['Kms_Driven'] = np.clip(df['Kms_Driven'], 0, kms_99)
print('Data cleaned and feature engineered!')

## 3. Train-Test Split and Preprocessing Pipeline

In [4]:
num_features = ['Car_Age', 'Present_Price', 'Kms_Driven', 'Owner']
cat_features = ['Fuel_Type', 'Seller_Type', 'Transmission', 'Brand']
X = df[num_features + cat_features]
y = df['Selling_Price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Training samples: {len(X_train)}, Testing samples: {len(X_test)}')

## 4. Model Training & Comparison

In [5]:
preprocessor = ColumnTransformer([
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), num_features),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), cat_features)
])

models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(max_depth=7, random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=150, max_depth=12, random_state=42)
}

results = []
for name, reg in models.items():
    pipe = Pipeline([('prep', preprocessor), ('model', reg)])
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)
    r2 = r2_score(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mae = mean_absolute_error(y_test, preds)
    results.append({'Algorithm': name, 'R2': round(r2, 4), 'RMSE': round(rmse, 4), 'MAE': round(mae, 4)})

pd.DataFrame(results)

## 5. Summary & Serialization
Random Forest achieved the best R² (>97%) with lowest RMSE and MAE, and is exported to `/models/best_car_price_model.joblib`.